# Working with the Catalog

The Catalog is a **SQLite file**, not a hosted database and not a live OSM
query. This notebook builds it from `tests/fixtures/catalog.osm`, then shows
how to reopen it, inspect tables, and use DayHike objects.

No Geofabrik download. No live Valhalla.


In [ ]:
import sqlite3
import tempfile
from pathlib import Path

from dayhike import Catalog, build_catalog, rank_dayhikes, resolve_seed

repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    fixture = candidate / "tests" / "fixtures" / "catalog.osm"
    if fixture.is_file():
        break
else:
    raise FileNotFoundError("tests/fixtures/catalog.osm")

path = Path(tempfile.mkdtemp()) / "catalog.sqlite"
built = build_catalog(fixture, path)
print("wrote", built.path)
print(built.ingest_report)


## Reopen from disk

`build_catalog` returns a Catalog already loaded. Later sessions only need
`Catalog.open` on the same path (for you, usually `data/catalog.sqlite`).


In [ ]:
catalog = Catalog.open(path)
assert catalog.path == built.path
assert [hike.id for hike in catalog.dayhikes] == [hike.id for hike in built.dayhikes]
print("open ok", [hike.name for hike in catalog.dayhikes])
print("labels table empty", sqlite3.connect(path).execute("SELECT COUNT(*) FROM labels").fetchone()[0])


## SQL you can run

Use this against `data/catalog.sqlite` after a real ingest, too. Schema:

- `dayhikes(id, name, length_m, difficulty)`
- `trailheads(dayhike_id, latitude, longitude)`
- `amenity_facts(dayhike_id, kind, count, nearest_m)`
- `feature_vectors(...)`
- `scaler(position, feature_name, mean, scale)`
- `labels` (empty on purpose)
- `catalog_meta(key, value)`


In [ ]:
connection = sqlite3.connect(path)
connection.row_factory = sqlite3.Row

print("meta", dict(connection.execute("SELECT key, value FROM catalog_meta")))
print("--- joined view ---")
rows = connection.execute(
    """
    SELECT d.id, d.name, d.length_m, d.difficulty,
           t.latitude, t.longitude,
           f.elevation_min_m, f.elevation_gain_m, f.difficulty_ordinal
    FROM dayhikes d
    JOIN trailheads t ON t.dayhike_id = d.id
    JOIN feature_vectors f ON f.dayhike_id = d.id
    ORDER BY d.id
    """
)
for row in rows:
    print(dict(row))

print("--- scaler ---")
for row in connection.execute(
    "SELECT position, feature_name, mean, scale FROM scaler ORDER BY position"
):
    print(dict(row))
connection.close()


## Typical Python session

Resolve a Seed you already hiked, rank the rest, inspect why a neighbor scored
as it did. Seed names must exist in the Catalog.


In [ ]:
seed_id = resolve_seed(catalog, "Valley Path")
seed = next(hike for hike in catalog.dayhikes if hike.id == seed_id)
print("seed", seed.name, "length_m", round(seed.length_m), "sac", seed.difficulty.sac_scale)

ranking = rank_dayhikes(catalog, ["Valley Path"])
for item in ranking.items:
    hike = item.dayhike
    print(
        "neighbor",
        hike.name,
        "score",
        round(item.score, 4),
        "length_m",
        round(hike.length_m),
        "sac",
        hike.difficulty.sac_scale,
        "hut_count",
        hike.trailhead.amenity_facts.hut.count,
    )
    print("  FeatureVector", hike.feature_vector)


## Next steps on a real extract

1. `uv run dayhike fetch-extracts --dest data/osm`
2. Optionally `osmium tags-filter` a country PBF (README).
3. `uv run dayhike ingest data/osm --catalog data/catalog.sqlite`
4. `SELECT name FROM dayhikes;` to see legal Seed names.
5. `uv run dayhike rank --catalog data/catalog.sqlite "That Name"`
6. DriveTime: `./data/valhalla/prepare-roads.sh` on an unfiltered extract,
   then `docker compose -f data/valhalla/compose.yaml up`, then `--origin`
   and `--max-hours`.
